# Factual Recall γ — Phase 1-4 (Standard 5 models bf16, hihp/hilp schema)

Models: Llama-3.1-8B-Inst, Gemma-2-9B-It, Qwen-2.5-7B-Inst, Qwen-2.5-14B-Inst, OLMo-2-13B-Inst.
Path: `model.model.layers[L].self_attn.o_proj` | bare `trace(prompt)` | bf16

**Cell labels**: hihp (hi-imp hi-pert, was A), hilp (hi-imp lo-pert, was B), C (lo-imp hi-pert), D (lo-imp lo-pert)
**Orderings (6)**: hihp_imp_desc, hihp_imp_asc, hilp_imp_desc, hilp_imp_asc, C_imp_asc, D_imp_asc


In [ ]:
# ── Cell 1: Drive mount → install → HF login ──
from google.colab import drive, runtime
drive.mount('/content/drive')

!pip install -q nnsight transformers accelerate scipy pandas scikit-learn

import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login as _hf_login
_hf_login(token=os.environ['HF_TOKEN'])
print('HF_TOKEN set + huggingface_hub.login() called.')

import json, gc, time, math, random, traceback
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=True)

log(f'Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    log(f'GPU: {torch.cuda.get_device_name(0)}  VRAM={torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')


In [ ]:
# ── Cell 2: Config ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_SPECS = [('meta-llama/Llama-3.1-8B-Instruct', 'llama-3.1-8b-instruct'), ('google/gemma-2-9b-it', 'gemma-2-9b-it'), ('Qwen/Qwen2.5-7B-Instruct', 'qwen-2.5-7b-instruct'), ('Qwen/Qwen2.5-14B-Instruct', 'qwen-2.5-14b-instruct'), ('allenai/OLMo-2-1124-13B-Instruct', 'olmo-2-13b-instruct')]

RATIOS = [0.05, 0.10, 0.20, 0.30, 0.50]
CELLS  = ['hihp', 'hilp', 'C', 'D', 'diag']   # hihp = hi-imp hi-pert, hilp = hi-imp lo-pert
ORDERINGS = [
    'hihp_imp_desc', 'hihp_imp_asc',
    'hilp_imp_desc', 'hilp_imp_asc',
    'C_imp_asc', 'D_imp_asc',
    'diag_rank_asc',   # NEW: C∪D pool, joint-rank selector
]   # 6 orderings: hi-imp cells get both desc+asc; lo-imp cells get asc only (conventional null)
REQUIRED_ORDERINGS = set(ORDERINGS)
N_TRIALS_TARGET = 80

DRIVE_BASE = '/content/drive/MyDrive/WCC'
DATA_DIR   = f'{DRIVE_BASE}/factual_recall/00_data'
OUT_BASE   = f'{DRIVE_BASE}/factual_recall'

assert os.path.exists(f'{DATA_DIR}/trial_definitions.csv'), \
    f'Phase 0 output missing — run factual_recall_data_prep.ipynb first'

all_trials = pd.read_csv(f'{DATA_DIR}/trial_definitions.csv')
log(f'Phase 0 trials: {len(all_trials)} across {all_trials["relation"].nunique()} relations')

n_per_rel = max(1, N_TRIALS_TARGET // all_trials['relation'].nunique())
sampled = (all_trials
    .groupby('relation', group_keys=False)
    .apply(lambda g: g.sort_values('trial_id').head(n_per_rel))
    .reset_index(drop=True))
if len(sampled) > N_TRIALS_TARGET:
    sampled = sampled.sample(N_TRIALS_TARGET, random_state=SEED).reset_index(drop=True)
trials_df = sampled.copy()
log(f'Sampled trials: {len(trials_df)} across {trials_df["relation"].nunique()} relations')


In [ ]:
# ── Cell 3: Common helpers ──
import nnsight as nns
from nnsight import LanguageModel
log(f'nnsight version: {nns.__version__}')

def get_token_id_with_space(tokenizer, text):
    prefix = 'The answer is'
    full = tokenizer.encode(prefix + ' ' + str(text), add_special_tokens=False)
    pref = tokenizer.encode(prefix, add_special_tokens=False)
    return full[len(pref)]

def unload_model(model):
    try: del model
    except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def ensure_dir(p):
    os.makedirs(p, exist_ok=True); return p

def phase_dir(model_short, phase):
    return ensure_dir(f'{OUT_BASE}/{model_short}/{phase}')

def phase_done(model_short, phase):
    return os.path.exists(f'{phase_dir(model_short, phase)}/config.json')

def mark_phase_done(model_short, phase, meta, start_time=None):
    payload = {**meta, 'completed_at': datetime.now().isoformat()}
    if start_time is not None:
        payload['elapsed_sec'] = round(time.time() - start_time, 2)
    with open(f'{phase_dir(model_short, phase)}/config.json', 'w') as f:
        json.dump(payload, f, indent=2)

def phase4_all_orderings_complete(model_short):
    """Returns True if behavioral_gamma.csv exists and covers all REQUIRED_ORDERINGS."""
    csv_path = f'{phase_dir(model_short, "30_patching")}/behavioral_gamma.csv'
    if not os.path.exists(csv_path):
        return False
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return False
    if 'ordering' not in df.columns:
        return False   # old schema without ordering column → treat as incomplete
    existing = set(df['ordering'].unique())
    return REQUIRED_ORDERINGS.issubset(existing)

def update_phase4_config(model_short, meta):
    """Update 30_patching/config.json preserving prior completed_at as first_completed_at_asc_only."""
    cfg_path = f'{phase_dir(model_short, "30_patching")}/config.json'
    if os.path.exists(cfg_path):
        try:
            cfg = json.load(open(cfg_path))
        except Exception:
            cfg = {}
    else:
        cfg = {}
    # Preserve prior completed_at if present (only A/B asc-only run)
    prev_completed = cfg.get('completed_at')
    if prev_completed and 'first_completed_at_asc_only' not in cfg:
        cfg['first_completed_at_asc_only'] = prev_completed
    # Update with new metadata
    cfg.update(meta)
    cfg['last_updated_at'] = datetime.now().isoformat()
    # Drop plain completed_at (replaced by last_updated_at); keep first_* for history
    cfg.pop('completed_at', None)
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f, indent=2)


In [ ]:
# ── Cell 4: Load + clean/collect/patch (STANDARD: Llama/Gemma-2/Qwen/OLMo, bf16, bare trace) ──

def load_model(model_id):
    log(f'Loading {model_id}  (bf16)')
    model = LanguageModel(model_id, dtype=torch.bfloat16, device_map='auto')
    cfg = model.config
    arch = {
        'num_layers' : cfg.num_hidden_layers,
        'num_heads'  : cfg.num_attention_heads,
        'head_dim'   : getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads),
        'hidden_size': cfg.hidden_size,
        'vocab_size' : cfg.vocab_size,
    }
    arch['total_heads'] = arch['num_layers'] * arch['num_heads']
    log(f'  arch: L={arch["num_layers"]} H={arch["num_heads"]} D={arch["head_dim"]} total={arch["total_heads"]}')
    return model, arch


def clean_logits_last(model, prompt):
    with model.trace(prompt) as tracer:
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


def collect_attn_input_per_layer(model, arch, prompt):
    L, H, D = arch['num_layers'], arch['num_heads'], arch['head_dim']
    saves = {}
    with model.trace(prompt) as tracer:
        for i in range(L):
            saves[i] = model.model.layers[i].self_attn.o_proj.input[0, -1, :].save()
    if len(saves) != L:
        raise RuntimeError(f'Trace failed: captured {len(saves)}/{L} layers')
    return np.stack([saves[i].detach().float().cpu().numpy().reshape(H, D) for i in range(L)])


def patched_logits_with_heads(model, arch, prompt, source_vec, patch_heads):
    H, D = arch['num_heads'], arch['head_dim']
    src = torch.tensor(source_vec, dtype=torch.bfloat16, device='cuda')
    sorted_patches = sorted(patch_heads, key=lambda lh: lh[0])
    with model.trace(prompt) as tracer:
        for layer_idx, head_idx in sorted_patches:
            start, end = head_idx * D, (head_idx + 1) * D
            model.model.layers[layer_idx].self_attn.o_proj.input[0, -1, start:end] = src[layer_idx, head_idx]
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


In [ ]:
# ── Cell 5: Phase 1-4 implementations ──
FAIL_LOG_CHARS = 500
FAIL_LIMIT_PER_TRIAL = 20

# ---------- Phase 1: Clean pass ----------
def run_clean_pass(model, arch, tokenizer, trials_df, model_short):
    out_path = f'{phase_dir(model_short, "10_collection")}/clean_logits.csv'
    rows = []; t0 = time.time()
    for i, trial in trials_df.iterrows():
        target_id     = get_token_id_with_space(tokenizer, trial['target'])
        competitor_id = get_token_id_with_space(tokenizer, trial['competitor'])
        try:
            lg = clean_logits_last(model, trial['prompt'])
            argmax_id = int(lg.argmax())
            sorted_idx = np.argsort(-lg)
            target_rank = int(np.where(sorted_idx == target_id)[0][0]) + 1
            target_logit = float(lg[target_id])
            competitor_logit = float(lg[competitor_id])
        except Exception as e:
            log(f'  Phase1 WARN trial={trial["trial_id"]}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
            argmax_id = -1; target_rank = -1
            target_logit = float('nan'); competitor_logit = float('nan')
        rows.append({
            'trial_id': trial['trial_id'], 'relation': trial['relation'],
            'subject': trial['subject'], 'target': trial['target'],
            'competitor': trial['competitor'],
            'target_id': target_id, 'competitor_id': competitor_id,
            'argmax_id': argmax_id,
            'argmax_token': tokenizer.decode([argmax_id]) if argmax_id >= 0 else '',
            'target_rank': target_rank, 'target_logit': target_logit,
            'competitor_logit': competitor_logit,
            'margin': target_logit - competitor_logit,
            'is_top1': (target_rank == 1),
            'is_top5': (1 <= target_rank <= 5),
        })
        if (i+1) % 20 == 0:
            log(f'  Phase1 {i+1}/{len(trials_df)}  elapsed={time.time()-t0:.0f}s')
    df = pd.DataFrame(rows); df.to_csv(out_path, index=False)
    log(f'  Phase1 complete  top1={int(df["is_top1"].sum())}  top5={int(df["is_top5"].sum())}  '
        f'elapsed={time.time()-t0:.0f}s')
    return df


# ---------- Phase 2: Importance + Perturbation per head ----------
def run_phase2_scoring(model, arch, tokenizer, trials_df, clean_df, model_short):
    out_imp  = f'{phase_dir(model_short, "20_scoring")}/importance_per_trial_head.csv'
    out_pert = f'{phase_dir(model_short, "20_scoring")}/perturbation_per_trial_head.csv'
    L, H = arch['num_layers'], arch['num_heads']
    clean_lookup = clean_df.set_index('trial_id')

    if os.path.exists(out_imp):
        imp_done = pd.read_csv(out_imp); done_imp = set(imp_done['trial_id'].unique())
        log(f'  Phase2 resume imp: {len(done_imp)} trials done')
    else:
        imp_done = pd.DataFrame(); done_imp = set()
    if os.path.exists(out_pert):
        pert_done = pd.read_csv(out_pert); done_pert = set(pert_done['trial_id'].unique())
    else:
        pert_done = pd.DataFrame(); done_pert = set()

    imp_rows  = list(imp_done.to_dict('records'))
    pert_rows = list(pert_done.to_dict('records'))
    t0 = time.time()

    for ti, trial in trials_df.iterrows():
        tid = trial['trial_id']
        if tid in done_imp and tid in done_pert:
            continue
        try:
            target_id = int(clean_lookup.loc[tid, 'target_id'])
            clean_target_logit = float(clean_lookup.loc[tid, 'target_logit'])
        except KeyError:
            log(f'  Phase2 skip {tid}: no clean_df entry')
            continue

        try:
            clean_cache   = collect_attn_input_per_layer(model, arch, trial['prompt'])
            corrupt_cache = collect_attn_input_per_layer(model, arch, trial['corrupt_prompt'])
        except Exception as e:
            log(f'  Phase2 FAIL collect {tid}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
            continue

        if tid not in done_pert:
            for l in range(L):
                for h in range(H):
                    l2 = float(np.linalg.norm(clean_cache[l, h] - corrupt_cache[l, h]))
                    pert_rows.append({'trial_id': tid, 'layer': l, 'head': h, 'perturbation_l2': l2})
            pd.DataFrame(pert_rows).to_csv(out_pert, index=False)

        if tid not in done_imp:
            tgt_logits_patched = np.full((L, H), np.nan)
            n_fail = 0; abort_trial = False
            for l in range(L):
                if abort_trial: break
                for h in range(H):
                    try:
                        lg = patched_logits_with_heads(model, arch, trial['prompt'], corrupt_cache, [(l, h)])
                        tgt_logits_patched[l, h] = float(lg[target_id])
                    except Exception as e:
                        if n_fail < 3:
                            log(f'    Phase2 imp FAIL tid={tid} L{l}H{h}: '
                                f'{type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
                        n_fail += 1
                        if n_fail >= FAIL_LIMIT_PER_TRIAL:
                            log(f'  Phase2 ABORT trial {tid} after {n_fail} head failures')
                            abort_trial = True; break
            for l in range(L):
                for h in range(H):
                    patched_logit = tgt_logits_patched[l, h]
                    delta = (patched_logit - clean_target_logit) if not np.isnan(patched_logit) else float('nan')
                    imp_rows.append({
                        'trial_id': tid, 'layer': l, 'head': h,
                        'clean_target_logit': clean_target_logit,
                        'patched_target_logit': patched_logit if not np.isnan(patched_logit) else float('nan'),
                        'delta_target_logit': delta,
                    })
            pd.DataFrame(imp_rows).to_csv(out_imp, index=False)

        elapsed = time.time() - t0
        log(f'  Phase2 trial {ti+1}/{len(trials_df)}  tid={tid}  '
            f'elapsed={elapsed:.0f}s  ({elapsed/max(ti+1,1):.0f}s/trial)')

    imp_df  = pd.DataFrame(imp_rows)
    pert_df = pd.DataFrame(pert_rows)

    imp_head = imp_df.groupby(['layer','head']).agg(
        mean_abs_delta=('delta_target_logit', lambda x: float(np.nanmean(np.abs(x)))),
        mean_signed_delta=('delta_target_logit', lambda x: float(np.nanmean(x))),
        n_trials=('trial_id', 'nunique'),
    ).reset_index()
    imp_head.to_csv(f'{phase_dir(model_short, "20_scoring")}/importance_per_head.csv', index=False)

    pert_head = pert_df.groupby(['layer','head']).agg(
        mean_perturbation_l2=('perturbation_l2', 'mean'),
        std_perturbation_l2=('perturbation_l2', 'std'),
        n_trials=('trial_id', 'nunique'),
    ).reset_index()
    pert_head.to_csv(f'{phase_dir(model_short, "20_scoring")}/perturbation_per_head.csv', index=False)
    log(f'  Phase2 complete  elapsed={time.time()-t0:.0f}s')
    return imp_head, pert_head


# ---------- Phase 3: Cell classification (median split, hihp/hilp/C/D) ----------
def classify_cells(imp_head, pert_head, model_short):
    merged = imp_head.merge(pert_head, on=['layer', 'head'])
    merged['importance']   = merged['mean_abs_delta']
    merged['perturbation'] = merged['mean_perturbation_l2']
    imp_med, pert_med = merged['importance'].median(), merged['perturbation'].median()
    def assign(row):
        hi_i = row['importance'] > imp_med
        hi_p = row['perturbation'] > pert_med
        if hi_i and hi_p:     return 'hihp'    # high-importance, high-perturbation (was 'A')
        if hi_i and not hi_p: return 'hilp'    # high-importance, low-perturbation (was 'B')
        if not hi_i and hi_p: return 'C'       # low-importance, high-perturbation (confound)
        return 'D'                              # low-importance, low-perturbation (inert)
    merged['cell'] = merged.apply(assign, axis=1)
    merged[['layer','head','importance','perturbation','cell']].to_csv(
        f'{phase_dir(model_short, "20_scoring")}/cell_classification.csv', index=False)
    for c in CELLS:
        n = (merged['cell']==c).sum()
        log(f'  Cell {c}: {n} heads ({n/len(merged)*100:.1f}%)')
    return merged


# ---------- Phase 4: γ group patching (6 orderings) ----------
def build_orderings(cell_df):
    """Build 6 orderings from cell_classification.
    hi-imp cells (hihp, hilp): both imp_desc (biggest-first) and imp_asc (smallest-first)
    lo-imp cells (C, D): imp_asc only (conventional null baseline)"""
    orderings = {}
    for cell_letter in ('hihp', 'hilp'):
        sub = cell_df[cell_df['cell']==cell_letter]
        heads_desc = list(zip(sub.sort_values('importance', ascending=False)['layer'].astype(int),
                              sub.sort_values('importance', ascending=False)['head'].astype(int)))
        heads_asc  = list(zip(sub.sort_values('importance', ascending=True)['layer'].astype(int),
                              sub.sort_values('importance', ascending=True)['head'].astype(int)))
        orderings[f'{cell_letter}_imp_desc'] = heads_desc
        orderings[f'{cell_letter}_imp_asc']  = heads_asc
    for cell_letter in ('C', 'D'):
        sub = cell_df[cell_df['cell']==cell_letter]
        heads_asc = list(zip(sub.sort_values('importance', ascending=True)['layer'].astype(int),
                             sub.sort_values('importance', ascending=True)['head'].astype(int)))
        orderings[f'{cell_letter}_imp_asc'] = heads_asc

    # NEW: diag_rank_asc — matched-size lower-left selector from C∪D pool
    from scipy.stats import rankdata as _rankdata
    _low_imp = cell_df[cell_df['cell'].isin(['C', 'D'])].reset_index(drop=True)
    if len(_low_imp) > 0:
        _imp_arr = _low_imp['importance'].to_numpy()        # already |imp| in FR
        _pert_arr = _low_imp['perturbation'].to_numpy()
        _joint_rank = _rankdata(_imp_arr) + _rankdata(_pert_arr)
        _ordered = _low_imp.iloc[np.argsort(_joint_rank)]
        orderings['diag_rank_asc'] = list(zip(
            _ordered['layer'].astype(int), _ordered['head'].astype(int)))
    else:
        orderings['diag_rank_asc'] = []
    return orderings


def run_gamma(model, arch, tokenizer, cell_df, trials_df, clean_df, model_short):
    out_path = f'{phase_dir(model_short, "30_patching")}/behavioral_gamma.csv'
    if os.path.exists(out_path):
        done = pd.read_csv(out_path)
        # Support migration: if old CSV has no 'ordering' column, skip resume (should not happen after migration)
        if 'ordering' in done.columns:
            done_keys = set(zip(done['trial_id'], done['ordering'], done['ratio'].round(4)))
            rows = list(done.to_dict('records'))
            log(f'  Phase4 resume: {len(done_keys)} (trial,ordering,ratio) triples done')
        else:
            log(f'  Phase4 WARN: existing CSV has no ordering column, starting fresh')
            done_keys = set(); rows = []
    else:
        done_keys = set(); rows = []

    total_heads = arch['total_heads']
    clean_lookup = clean_df.set_index('trial_id')
    orderings = build_orderings(cell_df)
    # Verify we have all required orderings
    missing = REQUIRED_ORDERINGS - set(orderings.keys())
    if missing:
        raise RuntimeError(f'Missing orderings: {missing}')

    # Cell mapping: each ordering maps back to its source cell
    ordering_to_cell = {}
    for cell_letter in ('hihp', 'hilp'):
        ordering_to_cell[f'{cell_letter}_imp_desc'] = cell_letter
        ordering_to_cell[f'{cell_letter}_imp_asc']  = cell_letter
    for cell_letter in ('C', 'D'):
        ordering_to_cell[f'{cell_letter}_imp_asc'] = cell_letter
    ordering_to_cell['diag_rank_asc'] = 'diag'

    t0 = time.time()
    for ti, trial in trials_df.iterrows():
        tid = trial['trial_id']
        try:
            target_id = int(clean_lookup.loc[tid, 'target_id'])
            competitor_id = int(clean_lookup.loc[tid, 'competitor_id'])
            clean_argmax_id = int(clean_lookup.loc[tid, 'argmax_id'])
            clean_target_logit = float(clean_lookup.loc[tid, 'target_logit'])
        except KeyError:
            log(f'  Phase4 skip {tid}: no clean entry'); continue

        # Only collect corrupt_cache if this trial has unfinished work
        trial_keys_needed = [(tid, o, round(r, 4)) for o in ORDERINGS for r in RATIOS]
        if all(k in done_keys for k in trial_keys_needed):
            continue

        try:
            corrupt_cache = collect_attn_input_per_layer(model, arch, trial['corrupt_prompt'])
        except Exception as e:
            log(f'  Phase4 FAIL collect {tid}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
            continue

        for ordering_name in ORDERINGS:
            cell_heads_sorted = orderings[ordering_name]
            cell_letter = ordering_to_cell[ordering_name]
            for ratio in RATIOS:
                key = (tid, ordering_name, round(ratio, 4))
                if key in done_keys:
                    continue
                k = max(1, int(total_heads * ratio))
                k_actual = min(k, len(cell_heads_sorted))
                patch_heads = cell_heads_sorted[:k_actual]
                try:
                    lg = patched_logits_with_heads(model, arch, trial['prompt'], corrupt_cache, patch_heads)
                    patched_argmax = int(lg.argmax())
                    sorted_idx = np.argsort(-lg)
                    patched_rank = int(np.where(sorted_idx == target_id)[0][0]) + 1
                    patched_target_logit = float(lg[target_id])
                    patched_competitor_logit = float(lg[competitor_id])
                except Exception as e:
                    log(f'    Phase4 WARN {tid} {ordering_name} r={ratio}: '
                        f'{type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
                    patched_argmax = -1; patched_rank = -1
                    patched_target_logit = float('nan'); patched_competitor_logit = float('nan')
                rows.append({
                    'trial_id': tid, 'cell': cell_letter, 'ordering': ordering_name, 'ratio': ratio,
                    'k': k, 'k_actual': k_actual,
                    'argmax_preserved': (clean_argmax_id == patched_argmax) if patched_argmax >= 0 else False,
                    'target_preserved_top1': (patched_argmax == target_id) if patched_argmax >= 0 else False,
                    'patched_argmax_id': patched_argmax,
                    'patched_target_rank': patched_rank,
                    'patched_target_logit': patched_target_logit,
                    'patched_competitor_logit': patched_competitor_logit,
                    'patched_margin': patched_target_logit - patched_competitor_logit,
                    'clean_target_logit': clean_target_logit,
                    'delta_target_logit': patched_target_logit - clean_target_logit,
                })
        pd.DataFrame(rows).to_csv(out_path, index=False)
        log(f'  Phase4 trial {ti+1}/{len(trials_df)} tid={tid}  elapsed={time.time()-t0:.0f}s')

    gamma_df = pd.DataFrame(rows)
    summary = gamma_df.groupby(['cell', 'ordering', 'ratio']).agg(
        argmax_stability=('argmax_preserved', 'mean'),
        target_top1_stability=('target_preserved_top1', 'mean'),
        mean_target_rank=('patched_target_rank', lambda x: float(x[x>=0].mean()) if (x>=0).any() else float('nan')),
        median_target_rank=('patched_target_rank', lambda x: float(x[x>=0].median()) if (x>=0).any() else float('nan')),
        mean_margin=('patched_margin', lambda x: float(np.nanmean(x))),
        mean_abs_delta=('delta_target_logit', lambda x: float(np.nanmean(np.abs(x)))),
        n_trials=('trial_id', 'count'),
    ).reset_index()
    summary.to_csv(f'{phase_dir(model_short, "30_patching")}/behavioral_gamma_summary.csv', index=False)
    log(f'  Phase4 complete  elapsed={time.time()-t0:.0f}s')
    return gamma_df, summary


In [ ]:
# ── Cell 6: Main loop ──
all_success = True
for MODEL_ID, MODEL_SHORT in MODEL_SPECS:
    log('=' * 70)
    log(f'START: {MODEL_SHORT}  ({MODEL_ID})')
    t_model = time.time(); model = None
    try:
        model, arch = load_model(MODEL_ID)
        tokenizer = model.tokenizer

        if phase_done(MODEL_SHORT, '10_collection'):
            log(f'  [Phase 1] SKIP'); clean_df = pd.read_csv(f'{phase_dir(MODEL_SHORT, "10_collection")}/clean_logits.csv')
        else:
            _t10=time.time(); log(f'  [Phase 1] clean pass ({len(trials_df)} trials)')
            clean_df = run_clean_pass(model, arch, tokenizer, trials_df, MODEL_SHORT)
            mark_phase_done(MODEL_SHORT, '10_collection',
                {'model_id': MODEL_ID, 'n_trials': len(clean_df),
                 'n_top1': int(clean_df['is_top1'].sum()),
                 'n_top5': int(clean_df['is_top5'].sum())},
                start_time=_t10)

        if phase_done(MODEL_SHORT, '20_scoring'):
            log(f'  [Phase 2] SKIP')
            imp_head = pd.read_csv(f'{phase_dir(MODEL_SHORT, "20_scoring")}/importance_per_head.csv')
            pert_head = pd.read_csv(f'{phase_dir(MODEL_SHORT, "20_scoring")}/perturbation_per_head.csv')
        else:
            _t20=time.time(); log(f'  [Phase 2] importance + perturbation per head')
            imp_head, pert_head = run_phase2_scoring(model, arch, tokenizer, trials_df, clean_df, MODEL_SHORT)

        cell_path = f'{phase_dir(MODEL_SHORT, "20_scoring")}/cell_classification.csv'
        if os.path.exists(cell_path):
            cell_df = pd.read_csv(cell_path)
            cell_values = set(cell_df['cell'].unique())
            if 'A' in cell_values or 'B' in cell_values:
                # Auto-migrate: old schema A/B → new hihp/hilp (prevents silent k_actual=0 bug in Phase 4)
                log(f'  [Phase 3] AUTO-MIGRATE cell_classification.csv: A→hihp, B→hilp')
                cell_df['cell'] = cell_df['cell'].replace({'A': 'hihp', 'B': 'hilp'})
                cell_df.to_csv(cell_path, index=False)
                log(f'    new cell counts: {cell_df["cell"].value_counts().to_dict()}')
                # Also invalidate any existing Phase 4 output (it was computed with wrong cell_df)
                bg_path = f'{phase_dir(MODEL_SHORT, "30_patching")}/behavioral_gamma.csv'
                if os.path.exists(bg_path):
                    bg = pd.read_csv(bg_path)
                    if 'ordering' in bg.columns:
                        invalid = bg[bg['ordering'].str.startswith(('hihp','hilp'), na=False)]
                        if len(invalid) > 0 and (invalid['k_actual'] == 0).any():
                            log(f'  [Phase 4] invalid rows detected (k_actual=0 for hihp/hilp), dropping {len(invalid)} rows')
                            bg_clean = bg[~bg['ordering'].str.startswith(('hihp','hilp'), na=False)]
                            bg_clean.to_csv(bg_path, index=False)
            else:
                log(f'  [Phase 3] SKIP (cell_classification.csv present with hihp/hilp schema)')
        else:
            log(f'  [Phase 3] cell classification (median split)')
            cell_df = classify_cells(imp_head, pert_head, MODEL_SHORT)

        if not phase_done(MODEL_SHORT, '20_scoring'):
            mark_phase_done(MODEL_SHORT, '20_scoring',
                {'model_id': MODEL_ID, 'n_heads': int(arch['total_heads'])},
                start_time=locals().get('_t20'))

        if phase4_all_orderings_complete(MODEL_SHORT):
            log(f'  [Phase 4] SKIP (all {len(REQUIRED_ORDERINGS)} orderings complete: {sorted(REQUIRED_ORDERINGS)})')
        else:
            _t40=time.time(); log(f'  [Phase 4] behavioral γ  cells={CELLS}  orderings={ORDERINGS}  ratios={RATIOS}')
            run_gamma(model, arch, tokenizer, cell_df, trials_df, clean_df, MODEL_SHORT)
            update_phase4_config(MODEL_SHORT,
                {'model_id': MODEL_ID, 'ratios': RATIOS, 'cells': CELLS, 'orderings': ORDERINGS,
                 'phase4_elapsed_sec': round(time.time()-_t40, 2)})

        log(f'  DONE: {MODEL_SHORT}  ({(time.time()-t_model)/60:.1f} min)')
    except Exception as e:
        log(f'  FAILED: {MODEL_SHORT}  {type(e).__name__}: {str(e)[:500]}')
        traceback.print_exc(); all_success = False
    finally:
        if model is not None:
            unload_model(model)

log('=' * 70)
log(f'ALL MODELS PROCESSED.  overall_success={all_success}')


In [ ]:
# ── Cell 7: Audio beep + runtime.unassign() only on success ──
try:
    from IPython.display import Audio, display
    sr = 44100; _t = np.linspace(0, 1, sr)
    display(Audio(0.5 * np.sin(2 * np.pi * 440 * _t), rate=sr, autoplay=True))
except Exception as e:
    log(f'beep failed: {e}')

if all_success:
    log('All models complete. Unassigning Colab runtime.')
    runtime.unassign()
else:
    log('Some models failed; keeping runtime alive for inspection.')
